In [1]:
from dotenv import load_dotenv
load_dotenv()

import sqlite3
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.sqlite import SqliteSaver

llm = init_chat_model("openai:gpt-4o")

conn = sqlite3.connect("memory.db", check_same_thread=False)

config={
    "configurable": {
        "thread_id": "11",
    },
    "recursion_limit": 2,
}

In [2]:
class State(MessagesState):
    pass

graph_builder = StateGraph(State)

In [3]:
def chatbot(state: State):
    response = llm.invoke(state["messages"])
    return { "messages": [response] }
    

In [4]:

graph_builder.add_node("chatbot", chatbot)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

graph = graph_builder.compile(
    checkpointer=SqliteSaver(conn)
)

In [7]:
result = graph.invoke(
    {
        "messages": [
            { "role": "user", "content": "I live in Korea now. And the city I live in is Suwon." }
        ]
    },
    config= config
)

In [8]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Hello, My name is SeongYun
================================== Ai Message ==================================

Hello SeongYun! How can I assist you today?
================================ Human Message =================================

I live in Korea now. And the city I live in is Suwon.
================================== Ai Message ==================================

That’s great! Suwon is a vibrant city with a lot of history, especially known for the Hwaseong Fortress. Is there something specific you’d like to know or discuss about Suwon, or is there anything else you need help with?


In [14]:
state_history = graph.get_state_history(config)

In [15]:
for state_snapshot in list(state_history):
    print(state_snapshot.next)
    print(state_snapshot.values["messages"])
    print("============\n")

()
[HumanMessage(content='Hello, My name is SeongYun', additional_kwargs={}, response_metadata={}, id='ee6b6273-de15-40ad-90fe-1e174459b20c'), AIMessage(content='Hello SeongYun! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 16, 'total_tokens': 29, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_6c2e0b41b2', 'id': 'chatcmpl-DNO5NfuzhrKuUHH9PLWYp9UFfRQEs', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019d2672-0737-7ab1-bdff-a47c662982f1-0', usage_metadata={'input_tokens': 16, 'output_tokens': 13, 'total_tokens': 29, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), HumanMessage(conte

In [18]:
state_history = graph.get_state_history(config)

to_fork = list(state_history)[-5]

print(to_fork.values["messages"])
print("=========================")
print(to_fork.config)

[HumanMessage(content='Hello, My name is SeongYun', additional_kwargs={}, response_metadata={}, id='ee6b6273-de15-40ad-90fe-1e174459b20c'), AIMessage(content='Hello SeongYun! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 16, 'total_tokens': 29, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_6c2e0b41b2', 'id': 'chatcmpl-DNO5NfuzhrKuUHH9PLWYp9UFfRQEs', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019d2672-0737-7ab1-bdff-a47c662982f1-0', usage_metadata={'input_tokens': 16, 'output_tokens': 13, 'total_tokens': 29, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), HumanMessage(content=

In [19]:
from langchain_core.messages import HumanMessage

graph.update_state(
    to_fork.config,
    {
        "messages": HumanMessage(
            content="I live in Korea now. And the city I live in is Gwanggyo.",
            id="d6d850ce-f39a-489d-a16f-53b27e550645",
        )
    }
)

{'configurable': {'thread_id': '11',
  'checkpoint_ns': '',
  'checkpoint_id': '1f128818-4ddf-6b28-8004-2fcd3cf78e2b'}}

In [21]:
forked_state = graph.get_state_history(
    {
        'configurable': {
            'thread_id': '11',
            'checkpoint_ns': '',
            'checkpoint_id': '1f128818-4ddf-6b28-8004-2fcd3cf78e2b'
        }
    }
)

list(forked_state)

[StateSnapshot(values={'messages': [HumanMessage(content='Hello, My name is SeongYun', additional_kwargs={}, response_metadata={}, id='ee6b6273-de15-40ad-90fe-1e174459b20c'), AIMessage(content='Hello SeongYun! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 16, 'total_tokens': 29, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_6c2e0b41b2', 'id': 'chatcmpl-DNO5NfuzhrKuUHH9PLWYp9UFfRQEs', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019d2672-0737-7ab1-bdff-a47c662982f1-0', usage_metadata={'input_tokens': 16, 'output_tokens': 13, 'total_tokens': 29, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reas

In [22]:
result = graph.invoke(
    None,
    {
        'configurable': {
            'thread_id': '11',
            'checkpoint_ns': '',
            'checkpoint_id': '1f128818-4ddf-6b28-8004-2fcd3cf78e2b'
        }
    }
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Hello, My name is SeongYun
================================== Ai Message ==================================

Hello SeongYun! How can I assist you today?
================================ Human Message =================================

I live in Korea now. And the city I live in is Gwanggyo.
================================== Ai Message ==================================

That's great! Gwanggyo is known for its modern urban planning and proximity to natural attractions. How do you find living there? Is there anything specific you'd like to know or discuss about Gwanggyo or Korea in general?
